# Oracle (V5) — Compléments d'évaluation pour le rapport de soutenance

Notebook **inférence seule** (aucun ré-entraînement). Répond à deux critiques du
rapport d'évaluation qui demandent des données non encore produites :

- **Critique 2.2** — *ablation two-stage sans DAG* : séparer la contribution du
  paradigme à deux étages de celle de la **structure fine** du DAG. On évalue, sur
  le **même** sous-ensemble ID (ACCESS-CM2), à K et pas de diffusion **identiques**,
  quatre variantes :
  - `oracle_normal` : `A_dag` appris (Oracle complet)
  - `oracle_random` : `A_dag` ← matrice acyclique aléatoire, **même norme de Frobenius**
    (teste si la *topologie apprise* compte, vs n'importe quelle matrice acyclique)
  - `oracle_zero`   : `A_dag` ← 0 (chemin causal coupé = propriété O3)
  - `noncausal`     : baseline CorrDiff (encodeur+RCN remplacés par un régresseur)

- **Critique 2.6** — *Q_int élargi* : passer de 2 à **6 perturbations contrôlées**
  incluant des **signes opposés** (test bidirectionnel : q −20 %, t −3 K) et une
  **échelle de magnitude** (q +40 %), ce qui répond à « plusieurs niveaux, plusieurs
  signes ».

**Prérequis** : mêmes checkpoints que `st_cdgm_v5_evaluation.ipynb`
(Oracle = `ckpt_v2_corrdiff_normal`, Noncausal = `ckpt_noncausal`). Le bootstrap
ci-dessous (Cellules 1–3, verbatim du notebook d'éval) charge tout automatiquement.

**Coût estimé A100** : ~25–35 min (ablation ~20 min à `N=180, K=12, 18 pas` ×3 runs
+ Q_int ~10 min à `N=30, K=2` ×6 perturbations ×2 modèles).

**Sorties** : `RESULTS_DIR/review_ablation_dag.json`, `RESULTS_DIR/review_qint_extended.json`
+ deux figures PNG.

In [ ]:
# >>> COLAB_BOOTSTRAP
# Bootstrap Colab optimisé — premier run ~3 min, re-runs ~30 s.
# Stratégie :
#   • Code sur SSD local (/content/) — git clone 5-10× plus rapide que vers Drive.
#   • Drive UNIQUEMENT pour les checkpoints (cf. cellule helpers plus bas).
#   • Pas de ``pip install -r requirements.txt`` brut (déclenche la compilation
#     CUDA de torch-scatter/torch-sparse → 20-30 min). À la place : install
#     pinned des seules deps non pré-installées par Colab.
#   • Wheels PyG pré-construits via le bon index (sinon torch-scatter/torch-sparse
#     compilent depuis les sources — interdit ici).
#
# Hors Colab : no-op.
import os, sys, subprocess, time, shlex
from pathlib import Path

# ── Configuration utilisateur ──────────────────────────────────────────
GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "two-stage-causal"  # hyperplan v2.0  # Phase 1 EDM rewrite (Karras 2022)
LOCAL_PROJECT = "/content/climate_data"  # SSD — toujours rapide
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True  # si ``import st_cdgm`` réussit déjà → skip pip

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    """Exécute une commande shell avec timing visible."""
    print(f"$ {cmd}")
    t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0
    print(f"  ↳ rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc


if _IS_COLAB:
    _T0 = time.time()
    print("🛰️  Colab détecté — bootstrap en cours…\n")

    # 1) Monter Drive (idempotent — pour la cellule de persistance plus loin)
    from google.colab import drive  # type: ignore[import-not-found]
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("   /content/drive déjà monté.")

    # 2) Clone vers SSD (PAS vers Drive — FUSE est lent pour des milliers de petits fichiers)
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        if GIT_URL is None:
            raise RuntimeError(
                "GIT_URL=None et projet absent du SSD. Renseignez GIT_URL ci-dessus, "
                "ou pré-uploadez le projet à " + LOCAL_PROJECT + " avant ce run."
            )
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")
    elif GIT_PULL_ON_RESUME:
        try:
            _run(f"git -C {LOCAL_PROJECT} pull --ff-only", timeout=60, check=False)
        except Exception as e:
            print(f"   ⚠️  git pull a levé : {e}")

    # 3) cd dans la racine — config/, src/, etc. en chemins relatifs
    os.chdir(project_path)
    print(f"   chdir → {os.getcwd()}\n")

    # 4) Test d'import — si st_cdgm marche déjà, on saute pip (énorme gain au re-run)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    _need_pip = True
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            _need_pip = False
            print("✓ Imports critiques OK — pip install sauté.")
        except ImportError as _imp_err:
            print(f"   import st_cdgm a échoué ({_imp_err}) — pip install requis.")

    if _need_pip:
        # 5) Versions PyTorch / CUDA déjà installées par Colab
        import torch
        TORCH_VER = torch.__version__.split("+")[0]  # ex. "2.5.1"
        TORCH_TAG = f"torch-{TORCH_VER}"             # ex. "torch-2.5.1"
        CUDA_TAG = "cu" + (torch.version.cuda or "121").replace(".", "") if torch.cuda.is_available() else "cpu"
        print(f"   torch={TORCH_VER}, cuda={CUDA_TAG}\n")

        # 6) Install des deps NON pré-installées par Colab
        # (numpy/pandas/scipy/sklearn/matplotlib/torch/torchvision/torchaudio/
        #  zarr/dask/xarray/h5netcdf/cartopy/tqdm/requests/ipykernel sont déjà là)
        EXTRA_DEPS = [
            "omegaconf==2.3.0",
            "hydra-core==1.3.2",
            "diffusers==0.36.0",
            "transformers==4.57.6",
            "accelerate==1.12.0",
            "huggingface-hub==0.36.0",
            "safetensors==0.7.0",
            "xbatcher",
            "webdataset",
            "cftime",
            "h5netcdf",
            "numcodecs",
            "torch-geometric",  # v2.3+ ne nécessite plus torch-scatter/torch-sparse
            "xformers",  # OOM fix R2 : memory-efficient attention sur sm_75 (T4)
        ]
        deps_str = " ".join(shlex.quote(p) for p in EXTRA_DEPS)
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location {deps_str}",
            timeout=600,
        )

        # 7) torch-scatter / torch-sparse — *optionnels* avec PyG ≥ 2.3 (fallback
        # pure-PyTorch). Décommentez si vous tombez sur un module qui les exige.
        # _PYG_INDEX = f"https://data.pyg.org/whl/{TORCH_TAG}+{CUDA_TAG}.html"
        # _run(
        #     f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
        #     f"torch-scatter torch-sparse -f {_PYG_INDEX}",
        #     timeout=600, check=False,
        # )

        # 8) Editable install du package — --no-deps pour ne PAS retomber sur
        # requirements.txt (qui réinstallerait torch et compilerait torch-scatter).
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
            f"--no-deps -e {LOCAL_PROJECT}",
            timeout=120,
        )

        # 9) Re-test des imports critiques
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            print("✓ st_cdgm importable.")
        except ImportError as e:
            print(f"⚠️  st_cdgm pas encore importable depuis ce kernel : {e}")
            print("   → Probablement un cache d'import — Runtime → Restart runtime puis re-run.")

    print(f"\n✅ Bootstrap Colab terminé en {time.time() - _T0:.1f}s.")

else:
    # Hors Colab : remonte automatiquement à la racine projet.
    _here = Path.cwd()
    for _candidate in [_here, *_here.parents]:
        if (_candidate / "config" / "training_config.yaml").exists() and (_candidate / "setup.py").exists():
            if _candidate != _here:
                os.chdir(_candidate)
                print(f"📂 chdir → {os.getcwd()} (racine projet détectée)")
            break
    print("ℹ️  Hors Colab — bootstrap sauté (assume install déjà faite).")

# === V5 specific smoke test (apres bootstrap) ===
try:
    from st_cdgm.models import ConditionalSkipBlock
    print("[Oracle smoke] ConditionalSkipBlock importable - Oracle features pretes")
except ImportError as e:
    print(f"[Oracle smoke] ConditionalSkipBlock NON disponible : {e}")
    print("           Verifier que la branche two-stage-causal contient src/st_cdgm/models/skip_direct.py")

try:
    from scripts.intervention_test import INTERVENTIONS
    print(f"[Oracle smoke] Phase 8 helpers OK ({len(INTERVENTIONS)} interventions)")
except ImportError as e:
    print(f"[Oracle smoke] scripts.intervention_test NON disponible : {e}")


In [ ]:
# === CONFIGURATION DES CHEMINS ===
from pathlib import Path

# Oracle (directory historiquement nomme ckpt_v2_corrdiff_normal)
V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")

# Noncausal = baseline CorrDiff générique
NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")

# Sorties
# Phase F (post-Oracle baseline, 2026-06-10) : nouveau dossier pour ne pas ecraser le baseline.
RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/oracle_evaluation")
V5_BASELINE_RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/v5_evaluation")  # baseline intact

# Phase F (2026-06-11) : evaluation TOUJOURS sur la version fine-tunee.
# Plus de selecteur conditionnel : ORACLE_DIR pointe toujours vers
# ORACLE_FINETUNED_DIR. Le baseline V5_DIR/ reste intact, lu UNIQUEMENT par
# Phase 6 (Cell 8) pour la comparaison FT vs baseline Oracle vs CorrDiff.
ORACLE_FINETUNED_DIR = Path("/content/drive/MyDrive/climate_data/oracle_finetuned")
ORACLE_DIR = ORACLE_FINETUNED_DIR
CHECKPOINT_NAME = "epoch_finetuned"

# Bootstrap (one-time) : si premier run et Phase F pas encore lancee, on
# copie le baseline Oracle pour que Cell 4 puisse booter le stack. Phase F
# (Cell 6) ecrasera ensuite ce fichier avec les vrais poids fine-tunes.
ORACLE_FINETUNED_DIR.mkdir(parents=True, exist_ok=True)
_ft_ckpt = ORACLE_FINETUNED_DIR / f"{CHECKPOINT_NAME}.pth"  # = epoch_finetuned.pth
_ft_last = ORACLE_FINETUNED_DIR / "epoch_last.pth"  # alias requis par finetune_bundle_b
_baseline_ckpt_src = V5_DIR / "epoch_last.pth"
if not _ft_ckpt.exists() or not _ft_last.exists():
    if _baseline_ckpt_src.exists():
        import shutil as _sh
        if not _ft_ckpt.exists():
            print(f"[Bootstrap] {_ft_ckpt.name} absent — copie depuis baseline pour Cell 4")
            _sh.copy(_baseline_ckpt_src, _ft_ckpt)
        if not _ft_last.exists():
            print(f"[Bootstrap] {_ft_last.name} absent — copie depuis baseline pour Phase F")
            _sh.copy(_baseline_ckpt_src, _ft_last)
        print(f"            (Phase F ecrasera les deux apres entrainement)")
    else:
        print(f"[ERREUR] Baseline introuvable : {_baseline_ckpt_src}")
        print(f"         Cell 4 va echouer. Verifie que V5_DIR contient le baseline.")
print(f"[Eval] ORACLE_DIR = {ORACLE_DIR.name}  CHECKPOINT_NAME = {CHECKPOINT_NAME}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Vérification existence
# Verifie existence des checkpoints actifs (ORACLE_DIR = FT, NONCAUSAL_DIR = baseline)
for label, p in [("Oracle", ORACLE_DIR), ("CorrDiff", NONCAUSAL_DIR)]:
    ckpt = p / f"{CHECKPOINT_NAME}.pth"
    fv = p / "final_validation_metrics.json"
    status_ckpt = "OK" if ckpt.exists() else "ABSENT"
    status_fv = "OK" if fv.exists() else "ABSENT"
    size_gb = ckpt.stat().st_size / 1e9 if ckpt.exists() else 0
    print(f"  {label:10s} : ckpt {status_ckpt} ({size_gb:.2f} GB)  metrics.json {status_fv}")
    print(f"             {p}")

print(f"\nRésultats -> {RESULTS_DIR}")

In [ ]:
# ============================================================
# Bootstrap autonome COMPLET (= Cells 15+16+17+30+32+40 du training notebook)
# Telecharge automatiquement TOUS les datasets manquants depuis Zenodo
# ============================================================
import os
import sys
import time
import json
import shutil
import urllib.request
import urllib.error
import torch
import numpy as np
from pathlib import Path
from omegaconf import OmegaConf

ON_COLAB = "google.colab" in sys.modules or Path("/content").exists()

# 1. CONFIG (base + override corrdiff_normal si dispo)
_base = Path("config/training_config.yaml")
_override = Path("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.load(_base)
if _override.exists():
    CONFIG = OmegaConf.merge(CONFIG, OmegaConf.load(_override))
    print("[OK] CONFIG = base + corrdiff_normal override")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
print(f"[OK] DEVICE={DEVICE}  lr_shape={lr_shape}  hr_shape={hr_shape}")

# 2. DATA_ROOT detection (= training cell 16)
DATA_ROOT_LOCAL = Path("data/raw")
DATA_ROOT_DRIVE = Path("/content/drive/MyDrive/climate_data/data")
_DATA_ROOT_LOCAL_SSD = Path("/content/data_local")

if ON_COLAB and DATA_ROOT_DRIVE.parent.parent.exists():
    DATA_ROOT = DATA_ROOT_DRIVE
    print(f"[INFO] DATA_ROOT = Drive ({DATA_ROOT})")
else:
    DATA_ROOT = DATA_ROOT_LOCAL
    print(f"[INFO] DATA_ROOT = local ({DATA_ROOT.resolve()})")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 3. BS32 SSD copy (Drive -> /content/data_local pour I/O rapide)
_BS32_ENABLED = bool(globals().get("DATA_LOCAL_SSD", True))
if ON_COLAB and _BS32_ENABLED and DATA_ROOT == DATA_ROOT_DRIVE:
    _files_to_copy = [
        ("train/predictor_ACCESS-CM2_hist.nc",   "predictor_ACCESS-CM2_hist.nc"),
        ("train/pr_ACCESS-CM2_hist.nc",          "pr_ACCESS-CM2_hist.nc"),
        ("static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc",
         "ERA5_eval_ccam_12km.198110_NZ_Invariant.nc"),
        ("normalization_coefs/mean_1974_2011.nc", "mean_1974_2011.nc"),
        ("normalization_coefs/std_1974_2011.nc",  "std_1974_2011.nc"),
    ]
    _ssd_train = _DATA_ROOT_LOCAL_SSD / "train"
    _ssd_static = _DATA_ROOT_LOCAL_SSD / "static_predictors"
    _ssd_norm = _DATA_ROOT_LOCAL_SSD / "normalization_coefs"
    for _d in (_ssd_train, _ssd_static, _ssd_norm):
        _d.mkdir(parents=True, exist_ok=True)
    _t_total = time.time()
    _bytes_copied = 0
    for _rel, _name in _files_to_copy:
        _src = DATA_ROOT_DRIVE / _rel
        if "train/" in _rel:
            _dst = _ssd_train / _name
        elif "static_predictors/" in _rel:
            _dst = _ssd_static / _name
        else:
            _dst = _ssd_norm / _name
        if not _src.exists():
            continue
        if _dst.exists() and _dst.stat().st_size == _src.stat().st_size:
            continue
        _t0 = time.time()
        print(f"   copie {_src.name}...", flush=True)
        shutil.copy2(_src, _dst)
        _bytes_copied += _dst.stat().st_size
        print(f"   OK {_dst.name} ({_dst.stat().st_size/1e6:.0f} MB en {time.time()-_t0:.1f}s)")
    if _bytes_copied > 0:
        print(f"BS32 SSD copy: {_bytes_copied/1e9:.2f} GB en {time.time()-_t_total:.1f}s")
    DATA_ROOT = _DATA_ROOT_LOCAL_SSD
    print(f"[INFO] DATA_ROOT redirige vers SSD : {_DATA_ROOT_LOCAL_SSD}")

# 4. Resolution des paths (= training cell 16 suite)
def _relocate(p):
    if not p:
        return p
    s = str(p)
    if s.startswith("data/raw/"):
        return str(DATA_ROOT / s[len("data/raw/"):])
    return s

for _key in ("lr_path", "hr_path", "static_path"):
    if CONFIG.data.get(_key):
        CONFIG.data[_key] = _relocate(CONFIG.data[_key])

LR_PATH = str(CONFIG.data.lr_path)
HR_PATH = str(CONFIG.data.hr_path)
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None
MEAN_PATH = str(DATA_ROOT / "normalization_coefs" / "mean_1974_2011.nc")
STD_PATH = str(DATA_ROOT / "normalization_coefs" / "std_1974_2011.nc")

URL_ZENODO_HR = "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1"
URL_ZENODO_LR = "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1"
URLS_TEST = [
    ("EC-Earth3_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_histupdated_compressed.nc?download=1"),
    ("EC-Earth3_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_historical_precip_compressed.nc?download=1"),
    ("NorESM2-MM_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_histupdated_compressed.nc?download=1"),
    ("NorESM2-MM_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_historical_precip_compressed.nc?download=1"),
]

# 5. stream_download (atomique + reprise + timeout)
def stream_download(url, dest, retries=5, chunk_size=1024*1024,
                    connect_timeout=30, read_timeout=120):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = urllib.request.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
            print(f"   reprise a {already/1e6:.1f} MB")
        try:
            with urllib.request.urlopen(req, timeout=connect_timeout) as resp:
                total = resp.length
                if total is None and resp.headers.get("Content-Length"):
                    total = int(resp.headers["Content-Length"])
                grand_total = (total + already) if total else None
                mode = "ab" if already > 0 else "wb"
                with open(part, mode) as f:
                    downloaded = already
                    last_log = time.time()
                    last_log_bytes = downloaded
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
                        downloaded += len(chunk)
                        now = time.time()
                        if now - last_log >= 5.0:
                            speed = (downloaded - last_log_bytes) / (now - last_log) / 1e6
                            if grand_total:
                                pct = 100.0 * downloaded / grand_total
                                print(f"     {downloaded/1e6:7.1f}/{grand_total/1e6:7.1f} MB ({pct:.0f}%) {speed:.1f} MB/s")
                            else:
                                print(f"     {downloaded/1e6:7.1f} MB {speed:.1f} MB/s")
                            last_log = now
                            last_log_bytes = downloaded
            os.replace(part, dest)
            print(f"   OK {dest.name} ({dest.stat().st_size/1e6:.0f} MB)")
            return True
        except urllib.error.HTTPError as e:
            if e.code in (503, 504, 429):
                wait = min(60, 2**attempt); print(f"   HTTP {e.code} retry {wait}s"); time.sleep(wait)
            elif e.code == 416:
                os.replace(part, dest); return True
            else:
                print(f"   HTTP {e.code}: {e.reason}"); return False
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2**attempt); print(f"   reseau retry {wait}s ({type(e).__name__})"); time.sleep(wait)
        except Exception as e:
            print(f"   ERREUR {type(e).__name__}: {e}"); return False
    return False

# 6. Telechargements train (HR, LR ACCESS-CM2)
if not Path(HR_PATH).exists():
    print(f"Download HR ACCESS-CM2: {HR_PATH}")
    if not stream_download(URL_ZENODO_HR, HR_PATH):
        raise RuntimeError("Echec download HR ACCESS-CM2")
if not Path(LR_PATH).exists():
    print(f"Download LR ACCESS-CM2: {LR_PATH}")
    if not stream_download(URL_ZENODO_LR, LR_PATH):
        raise RuntimeError("Echec download LR ACCESS-CM2")

# 7. Telechargements test (EC-Earth3, NorESM2-MM)
TEST_ROOT = DATA_ROOT / "test"
TEST_ROOT.mkdir(parents=True, exist_ok=True)
for _filename, _url in URLS_TEST:
    _filepath = TEST_ROOT / _filename
    if _filepath.exists():
        continue
    print(f"Download test: {_filename}")
    if not stream_download(_url, str(_filepath)):
        raise RuntimeError(f"Echec download {_filename}")

# 8. Statics + normalization (fallback gdown si Drive public)
_PUBLIC_DRIVE_FALLBACKS = {
    "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc":
        "1KY6IS1W5Wt-l_xyV7Qw8caA49zPzuSEx",
    "normalization_coefs/mean_1974_2011.nc":
        "14wVaJTUDgLwLlFcqRFA6pzJg9tZtAVQ0",
    "normalization_coefs/std_1974_2011.nc":
        "1ycqq9DqpfdOOiyQqgKs797OzRdHND3ZL",
}

def _gdown_install():
    try:
        import gdown; return True
    except ImportError:
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"], timeout=120)
            import gdown; return True
        except Exception:
            return False

def _try_gdown(path):
    if not path:
        return False
    pth = Path(path)
    rel_key = None
    for _key in _PUBLIC_DRIVE_FALLBACKS:
        if str(pth).endswith(_key.replace("/", os.sep)) or str(pth).endswith(_key):
            rel_key = _key; break
    if rel_key is None:
        return False
    file_id = _PUBLIC_DRIVE_FALLBACKS[rel_key]
    pth.parent.mkdir(parents=True, exist_ok=True)
    if not _gdown_install():
        return False
    import gdown
    try:
        print(f"   gdown.download(id={file_id}) -> {pth}")
        gdown.download(id=file_id, output=str(pth), quiet=False)
        return pth.exists() and pth.stat().st_size > 0
    except Exception as e:
        print(f"   gdown ERREUR: {e}"); return False

for _var, _name in [("STATIC_PATH", "Static"), ("MEAN_PATH", "Mean"), ("STD_PATH", "Std")]:
    _p = globals()[_var]
    if _p and Path(_p).exists():
        continue
    if _try_gdown(_p):
        print(f"   OK {_name} (gdown public)")
    else:
        print(f"   {_name} absent: {_p} -> None")
        globals()[_var] = None

# 9. Resume final
print()
print("Datasets disponibles :")
print(f"  LR train  : {LR_PATH}  ({'OK' if Path(LR_PATH).exists() else 'MISSING'})")
print(f"  HR train  : {HR_PATH}  ({'OK' if Path(HR_PATH).exists() else 'MISSING'})")
print(f"  Static    : {STATIC_PATH}  ({'OK' if STATIC_PATH and Path(STATIC_PATH).exists() else 'NONE'})")
print(f"  Mean/Std  : {MEAN_PATH} / {STD_PATH}")
for fname, _ in URLS_TEST:
    p = TEST_ROOT / fname
    print(f"  Test      : {p.name}  ({'OK' if p.exists() else 'MISSING'})")

# 10. GCM_REGISTRY pour OOD
GCM_REGISTRY = {
    "ACCESS-CM2":  (Path(LR_PATH), Path(HR_PATH), True),
    "EC-Earth3":   (TEST_ROOT / "EC-Earth3_histupdated_compressed.nc",
                     TEST_ROOT / "EC-Earth3_historical_precip_compressed.nc", False),
    "NorESM2-MM":  (TEST_ROOT / "NorESM2-MM_histupdated_compressed.nc",
                     TEST_ROOT / "NorESM2-MM_historical_precip_compressed.nc", False),
}

# 11. Pipeline + builder
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder

def make_pipeline(lr_path, hr_path):
    return NetCDFDataPipeline(
        lr_path=str(lr_path), hr_path=str(hr_path),
        static_path=str(STATIC_PATH) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        seq_len=int(CONFIG.data.seq_len),
        baseline_strategy=str(CONFIG.data.baseline_strategy),
        baseline_factor=int(CONFIG.data.baseline_factor),
        target_transform=str(CONFIG.data.get("target_transform", "log1p")),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
        precipitation_delta=float(CONFIG.data.get("precipitation_delta", 0.01)),
        lr_variables=list(CONFIG.data.lr_variables),
        hr_variables=list(CONFIG.data.hr_variables),
        static_variables=list(CONFIG.data.static_variables) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        means_path=str(MEAN_PATH) if MEAN_PATH and Path(MEAN_PATH).exists() else None,
        stds_path=str(STD_PATH) if STD_PATH and Path(STD_PATH).exists() else None,
        eager_load_datasets=bool(CONFIG.data.get("eager_load_datasets", False)),
    )

pipeline_access = make_pipeline(LR_PATH, HR_PATH)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline_access.get_static_dataset(),
    include_mid_layer=bool(CONFIG.graph.include_mid_layer),
)
print()
print(f"[OK] Builder cree ({len(builder.dynamic_node_types)} dyn + {len(builder.static_node_types)} static)")

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {"lr": lr_tensor, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero}

test_dataset = pipeline_access.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True,
)
sample = next(iter(test_dataset))
_runtime_dim = int(sample["lr"].shape[1])
if _runtime_dim != int(CONFIG.rcn.driver_dim):
    CONFIG.rcn.driver_dim = _runtime_dim
    CONFIG.rcn.reconstruction_dim = _runtime_dim
    print(f"[INFO] CONFIG.rcn.driver_dim -> {_runtime_dim}")

# 12. Stacks
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
from st_cdgm.models.edm_preconditioner import EDMConfig
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    ConditionalSkipBlock = None

def build_stack(ckpt_path, name):
    print(f"  [{name}] {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # 1. Encoder configs depuis CONFIG.encoder.metapaths (filtre selon allowed_nodes du builder)
    allowed_nodes = set(builder.dynamic_node_types + builder.static_node_types)
    encoder_configs = []
    for _mp in CONFIG.encoder.metapaths:
        _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
        if _src in allowed_nodes and _tgt in allowed_nodes:
            encoder_configs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_src, _rel, _tgt),
                pool=_mp.get("pool", "mean"),
            ))
    # Ajoute le static metapath si pipeline a static_dataset
    if pipeline_access.get_static_dataset() is not None:
        encoder_configs.append(IntelligibleVariableConfig(
            name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
        ))

    enc = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
    ).to(DEVICE)
    num_vars = len(encoder_configs)

    rcn_cell = RCNCell(
        num_vars=num_vars,
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=int(CONFIG.rcn.driver_dim),
        reconstruction_dim=int(CONFIG.rcn.reconstruction_dim),
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

    rh = GraphToGridDecoder(
        d_model=CONFIG.encoder.hidden_dim,
        hr_h=CONFIG.graph.hr_shape[0], hr_w=CONFIG.graph.hr_shape[1],
    ).to(DEVICE)

    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
    # Convertit unet_kwargs en dict + tuples pour block_types
    from omegaconf import OmegaConf as _OC
    _unet_kwargs = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ("down_block_types", "up_block_types"):
        if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
            _unet_kwargs[_k] = tuple(_unet_kwargs[_k])

    hr_channels = int(sample["residual"].shape[1])

    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=_unet_kwargs,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", False)),
        conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
        anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)
    def _safe_load(name, module):
        """Load state_dict avec checks None + dict-like + strip prefixes."""
        key = f"{name}_state_dict"
        if key not in ckpt:
            print(f"  [{name}] [WARN] {key} absent du checkpoint")
            return False
        sd = ckpt[key]
        if sd is None:
            print(f"  [{name}] [WARN] {key} is None - skip")
            return False
        if not hasattr(sd, "items"):
            print(f"  [{name}] [WARN] {key} not dict-like ({type(sd).__name__}) - skip")
            return False
        # Strip prefixes torch.compile ('_orig_mod.') et DDP ('module.')
        prefixes = ["_orig_mod.", "module."]
        stripped = {}
        for kk, vv in sd.items():
            new_k = kk
            for p in prefixes:
                if new_k.startswith(p):
                    new_k = new_k[len(p):]
            stripped[new_k] = vv
        try:
            missing, unexpected = module.load_state_dict(stripped, strict=False)
            if missing:
                print(f"  [{name}] [INFO] {len(missing)} keys manquantes (premieres : {missing[:3]})")
            if unexpected:
                print(f"  [{name}] [INFO] {len(unexpected)} keys inattendues (premieres : {unexpected[:3]})")
            return True
        except Exception as e:
            print(f"  [{name}] [ERREUR] load_state_dict: {type(e).__name__}: {e}")
            return False

    for n, m in [("encoder", enc), ("rcn_cell", rcn_cell),
                  ("regression_head", rh), ("diffusion", diff)]:
        _safe_load(n, m)
    skip = None
    if (SKIP_AVAILABLE and "skip_block_state_dict" in ckpt
            and ckpt["skip_block_state_dict"] is not None):
        skip = ConditionalSkipBlock(
            lr_channels=len(CONFIG.data.lr_variables),
            hr_shape=tuple(CONFIG.graph.hr_shape),
        ).to(DEVICE)
        try:
            skip.load_state_dict(ckpt["skip_block_state_dict"], strict=False)
            print(f"  [{name}] [+] skip_block ({skip.num_params()} params)")
        except Exception as e:
            print(f"  [{name}] [WARN] skip_block load failed: {e}")
            skip = None
    enc.eval(); rcn_cell.eval(); rh.eval(); diff.eval()
    if skip is not None:
        skip.eval()
    A_dag = rcn_cell.A_dag.detach().cpu().clone() if hasattr(rcn_cell, "A_dag") else None
    return {"encoder": enc, "rcn_runner": rcn_runner, "regression_head": rh,
            "diffusion": diff, "skip_block": skip, "A_dag": A_dag, "variant": name}

print()
print("Chargement des stacks...")
t0 = time.time()
# Phase F (2026-06-11) : ORACLE_DIR resolu via EVAL_VERSION (baseline / finetuned).
# Le baseline CorrDiff (NONCAUSAL_DIR) reste sur "epoch_last.pth" (jamais touche).
stack_v5 = build_stack(ORACLE_DIR / f"{CHECKPOINT_NAME}.pth", "Oracle")
stack_nc = build_stack(NONCAUSAL_DIR / "epoch_last.pth", "CorrDiff")
print(f"[OK] 2 stacks charges en {time.time()-t0:.1f}s")

# 13. Predict generique
@torch.no_grad()
def predict_with_stack(stack, batch, K=4, n_steps=32):
    enc, rcn, rh, diff, skip = (stack["encoder"], stack["rcn_runner"],
                                  stack["regression_head"], stack["diffusion"],
                                  stack["skip_block"])
    lr = batch["lr"].to(DEVICE)
    H_init = enc.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr[t] for t in range(lr.shape[0])]
    seq = rcn.run(H_init, drivers, reconstruction_sources=None)
    H_T = seq.states[-1]
    mu_c = rh(H_T)
    tshape = batch["residual"][-1].to(DEVICE).shape
    if tshape[-2:] != mu_c.shape[-2:]:
        mu_c = torch.nn.functional.interpolate(
            mu_c, size=tshape[-2:], mode="bilinear", align_corners=False,
        )
    if skip is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu, _ = skip(lr_last, mu_c)
    else:
        mu = mu_c
    mu = torch.nan_to_num(mu, nan=0.0)
    bl = batch["baseline"][-1].to(DEVICE)
    if bl.dim() == mu.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    ens = []
    for _ in range(K):
        o = diff.sample(
            conditioning=None, num_steps=n_steps,
            scheduler_type="edm_karras", apply_constraints=False,
            mu_HR=mu, baseline_log=bl,
        )
        r = o.residual if hasattr(o, "residual") else o
        ens.append((bl + mu + r).cpu())
    return torch.stack(ens, dim=0)

print()
print("=" * 70)
print("Bootstrap autonome COMPLET")
print("=" * 70)
print("Variables disponibles :")
print(f"  CONFIG, DEVICE, builder, convert_sample_to_batch, predict_with_stack")
print(f"  stack_v5, stack_nc, GCM_REGISTRY, make_pipeline")
print(f"  test_dataset (ACCESS-CM2 in-dist)")
print()
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"TEST_ROOT  : {TEST_ROOT}")


---
## Cellule 4 — Helpers probabilistes (verbatim du notebook d'éval, Phase 7)

`probabilistic_metrics(ens_log1p, truth_log1p)` convertit en mm/jour puis calcule
CRPS empirique, spread/skill, CRPS-SS, RMSE global, histogramme de rang. **Identique**
à celui qui a produit le Tableau VIII du mémoire → chiffres directement comparables.

In [ ]:
# ============================================================
# Phase 7 : runs OOD reels avec run_aligned_eval (vendored Rampal)
# + sidecar probabilistic_metrics : CRPS, RMSE, spread, CRPS-SS, rank histogram
# Sortie 1 : aligned_metrics_<GCM>_<variant>.json   (indices climatiques)
# Sortie 2 : probabilistic_metrics_<GCM>_<variant>.json (ensemble-based)
# ============================================================
import json
import time
import torch
import numpy as np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval

# Parametres
N_TIMES_OOD = 365
K_SAMPLES_OOD = 12          # bump 4 -> 12 pour CRPS empirique non bruite
N_STEPS_DIFF = 18

# --- Metriques probabilistes ---------------------------------------------

def _crps_empirical_fast(samples, obs):
    """CRPS empirique vectorise via tri (O(K log K) par point).

    samples : (K, ...) array, ensemble
    obs     : (...,) array, observation
    return  : (...,) array, CRPS par point
    Formule : E|X - y| - 0.5 * E|X - X'|, avec
              0.5 * (1/K^2) * sum_ij |xi - xj| = (1/K^2) * sum_k (2k - K - 1) * x_(k)
    """
    K = samples.shape[0]
    term1 = np.nanmean(np.abs(samples - obs[None]), axis=0)
    s = np.sort(samples, axis=0)
    k_idx = np.arange(1, K + 1).reshape((K,) + (1,) * (s.ndim - 1)).astype(np.float64)
    weights = 2.0 * k_idx - K - 1.0
    term2 = np.sum(weights * s, axis=0) / (K * K)
    return term1 - term2


def _crps_clim_per_pixel(truth):
    """CRPS de la climato empirique (distribution par pixel sur l'axe temps).

    Pour X, X' iid ~ distribution-truth(h,w) et y ~ idem :
        CRPS_clim(h,w) = E|X - y| - 0.5 * E|X - X'| = 0.5 * E|X - X'|
    (car E|X-Y| = E|X-X'| pour des copies iid).
    """
    T = truth.shape[0]
    t_sorted = np.sort(truth, axis=0)
    k_idx = np.arange(1, T + 1).reshape((T, 1, 1)).astype(np.float64)
    weights = 2.0 * k_idx - T - 1.0
    return np.sum(weights * t_sorted, axis=0) / (T * T)


def _rank_histogram(samples, truth):
    """Histogramme de Talagrand : rang de truth parmi les K samples (K+1 bins)."""
    K = samples.shape[0]
    rank = (samples < truth[None]).sum(axis=0).astype(np.int64)
    hist, _ = np.histogram(rank.flatten(), bins=np.arange(K + 2) - 0.5)
    return hist.astype(int).tolist()


def probabilistic_metrics(ens_log1p, truth_log1p):
    """Calcule toutes les metriques probabilistes apres conversion log1p -> mm/jour.

    ens_log1p   : (K, T, H, W) ensemble en espace log1p
    truth_log1p : (T, H, W) verite en espace log1p
    """
    ens = np.expm1(np.clip(ens_log1p.astype(np.float64), 0.0, None))
    truth = np.expm1(np.clip(truth_log1p.astype(np.float64), 0.0, None))

    pred_mean = ens.mean(axis=0)                         # (T, H, W)
    err2 = (pred_mean - truth) ** 2
    rmse_global = float(np.sqrt(np.nanmean(err2)))
    rmse_map_t = np.sqrt(np.nanmean(err2, axis=0))       # (H, W)

    ens_var = ens.var(axis=0)                            # (T, H, W)
    spread_global = float(np.sqrt(np.nanmean(ens_var)))
    spread_skill_ratio = float(spread_global / max(rmse_global, 1e-9))

    crps_model = _crps_empirical_fast(ens, truth)        # (T, H, W)
    crps_model_global = float(np.nanmean(crps_model))

    crps_clim_map = _crps_clim_per_pixel(truth)          # (H, W)
    crps_clim_global = float(np.nanmean(crps_clim_map))

    crps_ss = 1.0 - crps_model_global / max(crps_clim_global, 1e-9)

    hist = _rank_histogram(ens, truth)
    K = int(ens.shape[0])
    expected_per_bin = float(truth.size / (K + 1))
    chi2_uniform = float(sum((c - expected_per_bin) ** 2 / expected_per_bin for c in hist))

    return {
        "K_samples": K,
        "n_times": int(ens.shape[1]),
        "grid": [int(truth.shape[-2]), int(truth.shape[-1])],
        "rmse_global_mm": rmse_global,
        "rmse_map_mean_mm": float(np.nanmean(rmse_map_t)),
        "rmse_map_max_mm": float(np.nanmax(rmse_map_t)),
        "spread_global_mm": spread_global,
        "spread_skill_ratio": spread_skill_ratio,
        "crps_model_global_mm": crps_model_global,
        "crps_clim_global_mm": crps_clim_global,
        "crps_skill_score": float(crps_ss),
        "rank_histogram": hist,
        "rank_histogram_bins": list(range(len(hist))),
        "rank_histogram_chi2_vs_uniform": chi2_uniform,
        "_caveat": (
            "CRPS_clim computed from test-truth empirical distribution per pixel "
            "(includes the day under evaluation; slight optimistic bias for T~365). "
            "spread_skill_ratio ~1 = well-calibrated, <1 = under-dispersive, >1 = over-dispersive."
        ),
    }


---
## Critique 2.2 — Ablation *two-stage sans DAG*

On collecte un ensemble `(K, T, H, W)` sur le **même** sous-ensemble ID pour les
quatre variantes, avec les **mêmes** `N_TIMES / K / pas`. Le mode d'ablation est posé
via `rcn_cell.set_dag_ablation_mode(...)`, consommé dans le forward du RCN
(`causal_rcn.py:431`) — il **ne mutile pas** `A_dag.data`, il perturbe la matrice
masquée à la volée.

**Lecture attendue** :
- `noncausal` vs `oracle_zero` → apport du **two-stage causal** (architecture),
- `oracle_random` vs `oracle_normal` → apport de la **topologie apprise** (structure fine),
- `oracle_zero` vs `oracle_normal` → propriété **O3** (le chemin est-il porteur).

In [ ]:
# ============================================================
# Critique 2.2 — ablation DAG 4-way (inference-only, meme sous-ensemble ID)
# ============================================================
import json, time
import numpy as np
import torch
import matplotlib.pyplot as plt

ABL_N_TIMES = 40       # sous-ensemble ID ; baisser a 120 si le temps presse
ABL_K       = 8        # membres d'ensemble (comme Tableau VIII / phase7)
ABL_STEPS   = 14       # pas de diffusion (comme phase7)

rcn_cell_v5 = stack_v5["rcn_runner"].cell
assert hasattr(rcn_cell_v5, "set_dag_ablation_mode"), "RCNCell sans set_dag_ablation_mode : verifier la branche"
A_dag_orig = rcn_cell_v5.A_dag.detach().clone()

def collect_ensemble_id(stack, n_times, K, steps):
    """Ensemble (K,T,H,W) + truth (T,H,W) en espace log1p sur ACCESS-CM2 ID."""
    lr_path, hr_path, _ = GCM_REGISTRY["ACCESS-CM2"]
    pipe = make_pipeline(lr_path, hr_path)
    ds = pipe.build_sequence_dataset(seq_len=int(CONFIG.data.seq_len), stride=1, as_torch=True)
    ens_log, truth_log = [], []
    it = iter(ds)
    for i in range(n_times):
        try:
            s = next(it)
        except StopIteration:
            break
        b = convert_sample_to_batch(s, builder, DEVICE)
        ens = predict_with_stack(stack, b, K=K, n_steps=steps)   # (K,B,1,H,W)
        while ens.dim() > 3:
            ens = ens.squeeze(1)
        ens_log.append(ens.cpu().numpy())
        truth_log.append((b["baseline"][-1] + b["residual"][-1]).cpu().squeeze().numpy())
    return np.stack(ens_log, axis=1), np.stack(truth_log, axis=0)

# (nom, stack, mode d'ablation) — mode=None => baseline noncausal (pas de RCN causal)
ABL_VARIANTS = [
    ("oracle_normal", stack_v5, "normal"),
    ("oracle_random", stack_v5, "random"),
    ("oracle_zero",   stack_v5, "zero"),
    ("noncausal",     stack_nc, None),
]

abl_results = {}
t_glob = time.time()
print("=" * 74)
print(f"Ablation DAG 4-way | N={ABL_N_TIMES} K={ABL_K} steps={ABL_STEPS}")
print("=" * 74)
for name, stack, mode in ABL_VARIANTS:
    t0 = time.time()
    if mode is not None:
        rcn_cell_v5.set_dag_ablation_mode(mode if mode != "normal" else "normal", seed=42)
    print(f"\n[{name}] mode={mode} ...", flush=True)
    ens, truth = collect_ensemble_id(stack, ABL_N_TIMES, ABL_K, ABL_STEPS)
    prob = probabilistic_metrics(ens, truth)
    abl_results[name] = {
        "crps_mm": prob["crps_model_global_mm"],
        "crps_ss": prob["crps_skill_score"],
        "spread_skill_ratio": prob["spread_skill_ratio"],
        "rmse_mm": prob["rmse_global_mm"],
        "rank_chi2": prob["rank_histogram_chi2_vs_uniform"],
        "n_times": int(ens.shape[1]), "K": int(ens.shape[0]),
    }
    print(f"  CRPS={prob['crps_model_global_mm']:.4f}mm  CRPS-SS={prob['crps_skill_score']:+.4f}  "
          f"spread/skill={prob['spread_skill_ratio']:.4f}  RMSE={prob['rmse_global_mm']:.4f}mm  "
          f"({time.time()-t0:.0f}s)")

# restaure l'etat normal (par securite)
rcn_cell_v5.set_dag_ablation_mode("normal")
rcn_cell_v5.A_dag.data.copy_(A_dag_orig)

# --- Tableau recapitulatif ---
o = abl_results["oracle_normal"]; r = abl_results["oracle_random"]
z = abl_results["oracle_zero"];   n = abl_results["noncausal"]
print("\n" + "=" * 74)
print(f"{'Variante':<16}{'CRPS(mm)':>10}{'CRPS-SS':>10}{'spread/skill':>14}{'RMSE(mm)':>10}")
print("-" * 74)
for name in ["oracle_normal", "oracle_random", "oracle_zero", "noncausal"]:
    a = abl_results[name]
    print(f"{name:<16}{a['crps_mm']:>10.4f}{a['crps_ss']:>+10.4f}{a['spread_skill_ratio']:>14.4f}{a['rmse_mm']:>10.4f}")
print("-" * 74)
print("Lecture :")
print(f"  two-stage causal (noncausal -> oracle_zero)  : dCRPS = {n['crps_mm']-z['crps_mm']:+.4f} mm")
print(f"  topologie apprise (random -> normal)         : dCRPS = {r['crps_mm']-o['crps_mm']:+.4f} mm")
print(f"  propriete O3 (zero -> normal)                : dCRPS = {z['crps_mm']-o['crps_mm']:+.4f} mm")

out = RESULTS_DIR / "review_ablation_dag.json"
out.write_text(json.dumps({
    "protocol": {"n_times": ABL_N_TIMES, "K": ABL_K, "n_steps": ABL_STEPS,
                 "space": "mm/day (expm1 of log1p)", "gcm": "ACCESS-CM2 (ID)"},
    "variants": abl_results,
    "deltas": {
        "two_stage_causal_dCRPS_mm": n["crps_mm"] - z["crps_mm"],
        "learned_topology_dCRPS_mm": r["crps_mm"] - o["crps_mm"],
        "O3_dCRPS_mm": z["crps_mm"] - o["crps_mm"],
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[OK] {out}")

# --- Figure barres CRPS ---
fig, ax = plt.subplots(figsize=(8, 4.5))
names = ["oracle_normal", "oracle_random", "oracle_zero", "noncausal"]
disp = ["Oracle\n(A_dag appris)", "Oracle\n(A_dag aleatoire)", "Oracle\n(A_dag = 0)", "Noncausal\n(baseline)"]
vals = [abl_results[k]["crps_mm"] for k in names]
colors = ["#1a9850", "#66bd63", "#fdae61", "#d73027"]
bars = ax.bar(disp, vals, color=colors, edgecolor="black")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("CRPS (mm/jour) — plus bas = mieux")
ax.set_title("Ablation two-stage / DAG (ID ACCESS-CM2, meme protocole)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig_path = RESULTS_DIR / "review_ablation_dag.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.close()
print(f"[OK] {fig_path}")
print(f"\nAblation terminee en {(time.time()-t_glob)/60:.1f} min")


---
## Critique 2.6 — Q_int élargi (6 perturbations, signes opposés, échelle)

On passe de 2 à **6 perturbations** en unités physiques (dé-standardisées via les
stats LR, cf. protocole Phase 8) :

| # | Perturbation | Signe attendu | Ce que ça teste |
|---|---|---|---|
| 1 | q₈₅₀ +20 % | + | humidité ↑ → précip ↑ |
| 2 | q₈₅₀ −20 % | − | **test bidirectionnel** (humidité ↓ → précip ↓) |
| 3 | q₈₅₀ +40 % | + | **échelle de magnitude** |
| 4 | q₅₀₀ +20 % | + | humidité mid-niveau |
| 5 | t₈₅₀ +3 K | + | Clausius-Clapeyron |
| 6 | t₈₅₀ −3 K | − | **test bidirectionnel thermique** |

Chaque perturbation : `N=30` jours, `K=2`, bootstrap CI 95 %. `Q_int` = fraction de
signes corrects. Le test bidirectionnel est plus exigeant qu'un test unidirectionnel :
il vérifie que la réponse **change de sens** avec la perturbation.

In [ ]:
# ============================================================
# Critique 2.6 — Q_int elargi (6 perturbations, inference-only)
# ============================================================
import json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from scripts.intervention_test import apply_intervention

QINT_N = 20      # jours par perturbation
QINT_K = 2       # membres (comme phase8)
QINT_STEPS = 14

lr_vars = list(CONFIG.data.lr_variables)

# --- Standardisation physique (comme phase8) ---
standardization = None
try:
    pipe_stats = make_pipeline(GCM_REGISTRY["ACCESS-CM2"][0], GCM_REGISTRY["ACCESS-CM2"][1])
    raw_stats = pipe_stats.get_lr_stats()
    standardization = {}
    for v in lr_vars:
        mu = float(raw_stats["mean"][v].mean().values)
        sd = float(raw_stats["std"][v].mean().values)
        standardization[v] = {"mean": mu, "std": max(sd, 1e-12)}
    print(f"[OK] standardization : {len(standardization)}/{len(lr_vars)} variables (unites physiques)")
except Exception as e:
    print(f"[WARN] standardization indisponible ({e}) - signes peu fiables")

# --- Jeu de perturbations elargi ---
INTERV_EXT = [
    dict(name="q850_+20pct", variable_name="q_850", delta_type="multiplicative",   delta_value=1.20, expected_sign=+1),
    dict(name="q850_-20pct", variable_name="q_850", delta_type="multiplicative",   delta_value=0.80, expected_sign=-1),
    dict(name="q850_+40pct", variable_name="q_850", delta_type="multiplicative",   delta_value=1.40, expected_sign=+1),
    dict(name="q500_+20pct", variable_name="q_500", delta_type="multiplicative",   delta_value=1.20, expected_sign=+1),
    dict(name="t850_+3K",    variable_name="t_850", delta_type="additive_celsius", delta_value=+3.0, expected_sign=+1),
    dict(name="t850_-3K",    variable_name="t_850", delta_type="additive_celsius", delta_value=-3.0, expected_sign=-1),
]
for spec in INTERV_EXT:
    try:
        spec["variable_idx"] = lr_vars.index(spec["variable_name"])
    except ValueError:
        spec["variable_idx"] = None
        print(f"[SKIP] {spec['name']} : {spec['variable_name']} absent du LR")

def _boot_ci(vals, n_boot=1000, alpha=0.05):
    arr = np.asarray(vals, dtype=np.float64)
    if arr.size == 0:
        return None, None, None, None
    rng = np.random.default_rng(42)
    boots = np.array([rng.choice(arr, arr.size, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(arr.mean()), float(lo), float(hi), float((arr > 0).mean())

qint_out = {"V5": [], "Noncausal": []}
t_glob = time.time()
print("=" * 74)
print(f"Q_int elargi | {len([s for s in INTERV_EXT if s['variable_idx'] is not None])} perturbations x N={QINT_N} x K={QINT_K}")
print("=" * 74)

for spec in INTERV_EXT:
    if spec["variable_idx"] is None:
        continue
    dv5, dnc = [], []
    it = iter(test_dataset)
    for k in range(QINT_N):
        try:
            s = next(it)
        except StopIteration:
            break
        b = convert_sample_to_batch(s, builder, DEVICE)
        b_int = dict(b)
        b_int["lr"] = apply_intervention(b["lr"], spec, standardization=standardization)
        with torch.no_grad():
            pv0 = predict_with_stack(stack_v5, b,     K=QINT_K, n_steps=QINT_STEPS).nanmean(0).squeeze().cpu().numpy()
            pv1 = predict_with_stack(stack_v5, b_int, K=QINT_K, n_steps=QINT_STEPS).nanmean(0).squeeze().cpu().numpy()
            pn0 = predict_with_stack(stack_nc, b,     K=QINT_K, n_steps=QINT_STEPS).nanmean(0).squeeze().cpu().numpy()
            pn1 = predict_with_stack(stack_nc, b_int, K=QINT_K, n_steps=QINT_STEPS).nanmean(0).squeeze().cpu().numpy()
        dv5.append(float(np.nanmean(pv1 - pv0)))
        dnc.append(float(np.nanmean(pn1 - pn0)))
    for variant, d in [("V5", dv5), ("Noncausal", dnc)]:
        mean, lo, hi, fp = _boot_ci(d)
        sign_pred = int(np.sign(mean)) if mean not in (None, 0) else 0
        match = int(sign_pred == spec["expected_sign"])
        qint_out[variant].append({
            "intervention": spec["name"], "variable": spec["variable_name"],
            "expected_sign": spec["expected_sign"], "delta_mean": mean,
            "ci_lo": lo, "ci_hi": hi, "ci_excludes_zero": int(lo is not None and (lo > 0 or hi < 0)),
            "sign_pred": sign_pred, "match": match, "n": len(d),
        })
    print(f"  [{spec['name']:14s}] Oracle dm={qint_out['V5'][-1]['delta_mean']:+.5f} "
          f"({'OK' if qint_out['V5'][-1]['match'] else 'KO'})  |  "
          f"Noncausal dm={qint_out['Noncausal'][-1]['delta_mean']:+.5f} "
          f"({'OK' if qint_out['Noncausal'][-1]['match'] else 'KO'})")

q_v5 = float(np.mean([r["match"] for r in qint_out["V5"]])) if qint_out["V5"] else 0.0
q_nc = float(np.mean([r["match"] for r in qint_out["Noncausal"]])) if qint_out["Noncausal"] else 0.0
n_v5 = sum(r["match"] for r in qint_out["V5"]); n_tot = len(qint_out["V5"])
n_nc = sum(r["match"] for r in qint_out["Noncausal"])
print("\n" + "=" * 74)
print(f"Q_int Oracle    = {q_v5:.3f}  ({n_v5}/{n_tot} signes corrects)")
print(f"Q_int Noncausal = {q_nc:.3f}  ({n_nc}/{n_tot} signes corrects)")

out = RESULTS_DIR / "review_qint_extended.json"
out.write_text(json.dumps({
    "protocol": {"n_days": QINT_N, "K": QINT_K, "n_steps": QINT_STEPS,
                 "space_units": "physical (destandardized)"},
    "Q_int": {"Oracle": q_v5, "Noncausal": q_nc,
              "n_correct_oracle": n_v5, "n_correct_noncausal": n_nc, "n_total": n_tot},
    "interventions": qint_out,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[OK] {out}")

# --- Figure : deltas signes par perturbation ---
labels = [r["intervention"] for r in qint_out["V5"]]
dv = [r["delta_mean"] for r in qint_out["V5"]]
dn = [r["delta_mean"] for r in qint_out["Noncausal"]]
exp = [r["expected_sign"] for r in qint_out["V5"]]
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x - w/2, dv, w, label="Oracle", color="steelblue")
ax.bar(x + w/2, dn, w, label="Noncausal", color="orange")
for i, e in enumerate(exp):
    ax.annotate("attendu " + ("+" if e > 0 else "-"), (i, 0),
                textcoords="offset points", xytext=(0, -18 if max(dv[i], dn[i]) > 0 else 8),
                ha="center", fontsize=7, color="gray")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("delta pluie moyen (reponse a l'intervention)")
ax.set_title(f"Q_int elargi : Oracle {n_v5}/{n_tot} vs Noncausal {n_nc}/{n_tot} signes corrects")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig_path = RESULTS_DIR / "review_qint_extended.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.close()
print(f"[OK] {fig_path}")
print(f"\nQ_int elargi termine en {(time.time()-t_glob)/60:.1f} min")
